# Simglucose Simulation Results Notebook

## Instructions

[Comprehensive documentation here](Docs/simglucose.md) detailing how to run Trio oref algorithm variants through simglucose to conduct mechanistic in silico simulations.

### Run One Virtual Person

```
python3 simglucose/run_sim.py -u <virtual_patient> -a <alg_name> -d <days> -scen <meal_scenario_path> -fn <results_file>
```
* `virtual_patient`: Name of virtual patient. Valid patient names are age group followed by three digits. Age groups = [child, adolescent, adult]. Valid digits = [001, 002, 003, 004, 005, 006, 007, 008, 009, 010]
    * eg. adolescent002 or adult010
* `alg_name`: Optional argument. Name of the Javascript oref algorithm variant to run instead of the Swift algorithm. Defaults to `"swift"` if no argument given. Possible choices:
    * `jsbug`: Original Javascript implementation.
    * `js`: Javascript implementation of bug-free Swift oref algorithm.
    * `swift`: Swift implementation of oref algorithm.
* `days`: Number of days simulation runs (must be whole number)
* `meal_scenario_path`: Optional argument. Filepath to precomputed .npy file containing meal scenario.
* `results_file`: Filepath to csv file where outputs will be written to.

### Run All Virtual People
Run the following commands to execute this [script](./Scripts/run_simglucose.sh) which simulates all 30 virtual persons. There is the option to run simulations in parallel in independent processes. Set `PARALLELISM` to the number of processes you want to run concurrently. 

```shell
chmod +x Scripts/run_simglucose.sh # if have not run script yet
./Scripts/run_simglucose.sh
```

## Code

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
children = [f"child{n}" for n in ["001","002","003","004","005","006","007","008","009","010"]]
adolescents = [f"adolescent{n}" for n in ["001","002","003","004","005","006","007","008","009","010"]]
adults = [f"adult{n}" for n in ["001","002","003","004","005","006","007","008","009","010"]]
all_users = children + adolescents + adults

results_folder = "simglucoseResults"

In [ ]:
def glycemia_risk_index(tar, tbr, tvar, tvbr):
    gri = 3 * tvbr + 2.4 * tbr + 1.6 * tvar + 0.8 * tar
    return 100.0 if gri > 100.0 else gri

def glucose_stats(glucose):
    low_bound = 70
    v_low_bound = 54
    v_high_bound = 250
    high_bound = 180
    glucose = np.array(glucose)

    tar = 100 * np.average(glucose > high_bound)
    tvar = 100 * np.average(glucose > v_high_bound)
    tbr = 100 * np.average(glucose < low_bound)
    tvbr = 100 * np.average(glucose < v_low_bound)
    tir = 100 - tar - tbr

    gri = glycemia_risk_index(tar, tbr, tvar, tvbr)

    return tir, tar, tbr, tvar, tvbr, gri

In [ ]:
def user_results(user:str, alg:str):
    path = f"{results_folder}/{user}/{alg}.csv"
    glucose = pd.read_csv(path)['CGM'].to_list()
    return glucose_stats(glucose)

def age_group_results(group:list, alg:str):
    results = {"User":[], "TIR":[], "TAR":[], "TBR":[], "TVAR":[], 
                     "TVBR":[], "GRI":[]}
    
    for user in group:
        tir, tar, tbr, tvar, tvbr, gri = user_results(user, alg)
        results["User"].append(user)
        results["TIR"].append(tir)
        results["TAR"].append(tar)
        results["TBR"].append(tbr)
        results["TVAR"].append(tvar)
        results["TVBR"].append(tvbr)
        results["GRI"].append(gri)

    for category, values in results.items():
        if category == "User": 
            results["User"].append("average")
        else:
            avg = np.average(results[category]).item()
            results[category].append(avg)

    return results

In [ ]:
print('Children Swift oref Simulation Results')
df = pd.DataFrame.from_dict(age_group_results(children, 'swift'))
display(df.style.hide())

print('Children Buggy Javascript oref Simlation Results')
df = pd.DataFrame.from_dict(age_group_results(children, 'jsbug'))
display(df.style.hide())

In [ ]:
print('Adolescent Swift oref Simulation Results')
df = pd.DataFrame.from_dict(age_group_results(adolescents, 'swift'))
display(df.style.hide())

print('Adolescent Buggy Javascript oref Simulation Results')
df = pd.DataFrame.from_dict(age_group_results(adolescents, 'jsbug'))
display(df.style.hide())

In [ ]:
print('Adult Swift oref Simulation Results')
df = pd.DataFrame.from_dict(age_group_results(adults, 'swift'))
display(df.style.hide())

print('Adult Buggy Javascript oref Simulation Results')
df = pd.DataFrame.from_dict(age_group_results(adults, 'jsbug'))
display(df.style.hide())

In [ ]:
print('All Virtual Persons Swift oref Simulation Results')
df = pd.DataFrame.from_dict(age_group_results(all_users, 'swift'))
display(df.style.hide())

print('All Virtual Persons Buggy Javascript oref Simulation Results')
df = pd.DataFrame.from_dict(age_group_results(all_users, 'jsbug'))
display(df.style.hide())